# Challenge 05 - Digit Recognition (MLP)

Recognition of the digit (0-9) pronounced in each audio track with MLPs (`Dense` and `Dropout` layers only) trained with TensorFlow / Keras. The input comes from the MFCCs without temporal collapse. Two kinds of networks are combined:

- **crop networks**: the speech part of the track is cropped using the energy (c0) and resampled to a fixed number of time steps. Each track is cropped in several ways and all the crops are used as training patterns;
- **frame networks**: every high-energy frame of the track is a pattern, described by the MFCCs of the frames around it, and the network classifies every frame.

The final classifier averages the output probabilities of the two groups. Each test track is classified independently of the others. The predictions are saved in `Es7Predictions.txt`.

## Setup

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import numpy as np
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es7Predictions.txt'

# Crop networks: (mode, threshold) of the crops of each track
smooth_width = 5        # moving average (frames) applied to c0 before the speech detection
min_speech_frames = 8   # shorter detections are discarded (the whole track is used)
n_steps = 16            # time steps of the resampled crop
CROPS = [('span', 0.55), ('span', 0.65), ('span', 0.75), ('span', 0.85),
         ('peak', 0.40), ('peak', 0.55), ('peak', 0.70)]
SINGLE_CROP = [('span', 0.75)]   # reference: one crop per track
n_crop_models = 5
CROP_NET = dict(hidden=(256, 128), dropout_in=0.2, l2=1e-3, smoothing=0.0, epochs=100, batch=32)

# Frame networks: one group of networks per fraction of frames kept
FRAME_KEEPS = [0.20, 0.25, 0.30]   # fraction of the frames with the highest c0
min_kept_frames = 10               # minimum number of frames kept for a track
context = np.array([1, 2, 3, 4, 6, 8, 12, 16, 24, 32])   # frames at +-context around the frame
n_frame_models = 5
FRAME_NET = dict(hidden=(512, 512), dropout_in=0.1, l2=1e-4, smoothing=0.1, epochs=20, batch=256)

# MLP (common)
dropout_hidden = 0.4
learning_rate = 1e-3

seed = 1234

## Helper Functions

Dataset loading, feature extraction, model definition, training, prediction and export.

Everything is computed from the track alone.

Crop features:

- **Speech detection**: c0 is smoothed and compared with the relative threshold `min(c0) + frac * (max(c0) - min(c0))`, which adapts to the noise level of each recording. In `span` mode the speech goes from the first to the last frame above the threshold; in `peak` mode it is the contiguous region around the energy peak.
- **Normalization**: the mean of c1-c19 over the speech is subtracted (cepstral mean normalization) and c0 is replaced by c0 minus its maximum.
- **Resampling**: the speech segment is linearly interpolated to `n_steps` frames and flattened; the log of its length is appended. The best threshold changes with the noise, so each training track is cropped with all the `CROPS` and every crop is a training pattern with the label of the track. For a test track the probabilities of its own crops are averaged.

Frame features:

- **Frame selection**: the `keep` fraction of the frames with the highest c0 is kept (at least `min_kept_frames`). It is based on the rank, so it does not depend on the level of the noise, and the frames do not need to be contiguous.
- **Normalization**: c1-c19 are standardized with the mean and the standard deviation of the kept frames and c0 is replaced by c0 minus its maximum.
- **Context**: the coefficients of the frames at the offsets `-context`, 0, `+context` are stacked (the spacing grows with the distance), and the relative position of the frame between the first and the last kept frame is appended.
- Every kept frame is a training pattern with the label of the track. The probability of a track is the softmax of the mean log-probability of its frames.

The scalers are fitted on the training patterns only.

In [2]:
def load_dataset(db_path, filename):
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def speech_bounds(m, frac, mode):
    pad = smooth_width // 2
    c0 = np.convolve(np.pad(m[:, 0], pad, mode='edge'), np.ones(smooth_width) / smooth_width, mode='valid')
    thr = c0.min() + frac * (c0.max() - c0.min())
    if mode == 'span':
        idx = np.where(c0 >= thr)[0]
        lo, hi = idx[0], idx[-1]
    else:
        lo = hi = c0.argmax()
        while lo > 0 and c0[lo - 1] >= thr:
            lo -= 1
        while hi < len(c0) - 1 and c0[hi + 1] >= thr:
            hi += 1
    if hi - lo + 1 < min_speech_frames:
        return 0, len(m) - 1
    return lo, hi


def resample(m, n):
    old, new = np.linspace(0, 1, len(m)), np.linspace(0, 1, n)
    return np.stack([np.interp(new, old, m[:, j]) for j in range(m.shape[1])], axis=1)


def crop_features(m, mode, frac):
    m = np.asarray(m, dtype=float)
    lo, hi = speech_bounds(m, frac, mode)
    seg = m[lo:hi + 1]
    seg = np.hstack([seg[:, :1] - seg[:, 0].max(), seg[:, 1:] - seg[:, 1:].mean(axis=0)])
    return np.append(resample(seg, n_steps).ravel(), np.log(len(seg)))


def frame_features(m, keep):
    m = np.asarray(m, dtype=float)
    t = len(m)
    n_keep = min(t, max(min_kept_frames, int(round(keep * t))))
    kept = np.zeros(t, dtype=bool)
    kept[np.argsort(-m[:, 0])[:n_keep]] = True
    coef = (m[:, 1:] - m[kept, 1:].mean(axis=0)) / (m[kept, 1:].std(axis=0) + 1e-6)
    x = np.hstack([m[:, :1] - m[:, 0].max(), coef])
    offsets = np.concatenate([-context[::-1], [0], context])
    reach = context.max()
    x = np.pad(x, ((reach, reach), (0, 0)), mode='edge')
    x = x[np.arange(t)[:, None] + reach + offsets].reshape(t, -1)
    on = np.flatnonzero(kept)
    position = np.clip((np.arange(t) - on[0]) / max(on[-1] - on[0], 1), 0, 1)
    return np.hstack([x, position[:, None]])[kept]


def build_model(n_features, net):
    reg = tf.keras.regularizers.l2(net['l2'])
    layers = [tf.keras.layers.Input(shape=(n_features,)), tf.keras.layers.Dropout(net['dropout_in'])]
    for units in net['hidden']:
        layers.append(tf.keras.layers.Dense(units, activation='relu', kernel_initializer='he_normal', kernel_regularizer=reg))
        layers.append(tf.keras.layers.Dropout(dropout_hidden))
    layers.append(tf.keras.layers.Dense(10, activation='softmax'))
    model = tf.keras.Sequential(layers)
    model.compile(loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=net['smoothing']),
                  optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


def fit_predict(x_tr, y_tr, x_te_list, n, net):
    scaler = StandardScaler().fit(x_tr)
    x_tr = scaler.transform(x_tr)
    x_te_list = [scaler.transform(x) for x in x_te_list]
    y_tr = tf.keras.utils.to_categorical(y_tr, 10)
    proba = [0] * len(x_te_list)
    for i in range(n):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed + i)
        model = build_model(x_tr.shape[1], net)
        model.fit(x_tr, y_tr, epochs=net['epochs'], batch_size=net['batch'], shuffle=True, verbose=0)
        for j, x in enumerate(x_te_list):
            proba[j] = proba[j] + model.predict(x, verbose=0, batch_size=4096) / n
    return proba


def crop_proba(m_tr, y_tr, m_te, crops):
    x_tr = np.vstack([[crop_features(m, mode, frac) for m in m_tr] for mode, frac in crops])
    x_te = [np.array([crop_features(m, mode, frac) for m in m_te]) for mode, frac in crops]
    proba = fit_predict(x_tr, np.tile(y_tr, len(crops)), x_te, n_crop_models, CROP_NET)
    return sum(proba) / len(crops)


def frame_proba(m_tr, y_tr, m_te):
    total = 0
    for keep in FRAME_KEEPS:
        f_tr = [frame_features(m, keep) for m in m_tr]
        f_te = [frame_features(m, keep) for m in m_te]
        labels = np.repeat(y_tr, [len(f) for f in f_tr])
        track = np.repeat(np.arange(len(f_te)), [len(f) for f in f_te])
        proba = fit_predict(np.vstack(f_tr), labels, [np.vstack(f_te)], n_frame_models, FRAME_NET)[0]
        logp = np.zeros((len(f_te), 10))
        np.add.at(logp, track, np.log(proba + 1e-6))
        logp = logp / np.bincount(track)[:, None]
        e = np.exp(logp - logp.max(axis=1, keepdims=True))
        total = total + e / e.sum(axis=1, keepdims=True)
    return total / len(FRAME_KEEPS)


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains sessions 1 and 4 (620 tracks), the test set sessions 2 and 3 (620 tracks, without labels). The label is the pronounced digit (`digits`). The speaker (`person_ids`) and the compact `features` are not used: only the `mfccs`.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_m = list(train_data['mfccs'])
test_m = list(test_data['mfccs'])
train_y = np.array(train_data['digits'], dtype=np.int32)
train_sess = np.array(train_data['session_ids'])

print('Training tracks: %d, test tracks: %d' % (len(train_y), len(test_m)))
print('Tracks per digit:', np.bincount(train_y))
print('Frames per track: %d-%d' % (min(len(m) for m in train_m), max(len(m) for m in train_m)))

Training tracks: 620, test tracks: 620
Tracks per digit: [62 62 62 62 62 62 62 62 62 62]
Frames per track: 79-745


## Model: Ensemble of MLPs

All the networks are made of a `Dropout` on the input, two `Dense` layers (ReLU, He initialization, L2 regularization) each followed by `Dropout(0.4)`, and a softmax output with 10 units. They are trained with Adam for a fixed number of epochs (there is no early stopping because the validation data are other sessions, not a random hold-out).

- Crop networks: 5 networks with different seeds, 321 inputs (16 time steps x 20 coefficients + log length), 256 and 128 units, `Dropout(0.2)` on the input.
- Frame networks: 5 networks for each of the 3 fractions of kept frames, 421 inputs (21 frames x 20 coefficients + position), 512 and 512 units, `Dropout(0.1)` on the input, label smoothing. They are larger because they are trained on many more patterns.

The final prediction is the argmax of the mean of the two probabilities.

In [4]:
n_crop = crop_features(train_m[0], *CROPS[0]).shape[0]
frames = [frame_features(m, FRAME_KEEPS[0]) for m in train_m]
print('Crop network inputs: %d, training patterns: %d' % (n_crop, len(train_m) * len(CROPS)))
print('Frame network inputs: %d, training patterns (keep %.2f): %d' % (frames[0].shape[1], FRAME_KEEPS[0], sum(len(f) for f in frames)))
build_model(frames[0].shape[1], FRAME_NET).summary()

Crop network inputs: 321, training patterns: 4340
Frame network inputs: 421, training patterns (keep 0.20): 38709


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dropout (Dropout)               │ (None, 421)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       216,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 483,850 (1.85 MB)

 Trainable params: 483,850 (1.85 MB)

 Non-trainable params: 0 (0.00 B)

## Model Evaluation

The split is done by session to simulate new recording conditions: train on one session and validate on the other, in both directions. The crop networks with a single crop per track are reported as a reference, to show the effect of training on several crops. The choices (crops, kept frames, context, combination) were made on this same validation, so the final accuracy is slightly optimistic.

In [5]:
cv = {}
for a, c in [(1, 4), (4, 1)]:
    m_tr = [m for m, s in zip(train_m, train_sess) if s == a]
    m_va = [m for m, s in zip(train_m, train_sess) if s == c]
    y_tr, y_va = train_y[train_sess == a], train_y[train_sess == c]
    single = crop_proba(m_tr, y_tr, m_va, SINGLE_CROP)
    crop = crop_proba(m_tr, y_tr, m_va, CROPS)
    frame = frame_proba(m_tr, y_tr, m_va)
    cv[(a, c)] = (y_va, {'crop networks, single crop': single, 'crop networks': crop,
                         'frame networks': frame, 'crop + frame networks': (crop + frame) / 2})
    print('Train session %d -> validation session %d done' % (a, c))

Train session 1 -> validation session 4 done
Train session 4 -> validation session 1 done


In [6]:
names = list(cv[(1, 4)][1])
for name in names:
    accs = [accuracy_score(y, proba[name].argmax(axis=1)) for y, proba in cv.values()]
    print('%-28s 1->4: %.4f   4->1: %.4f   mean: %.4f' % (name, accs[0], accs[1], np.mean(accs)))

crop networks, single crop   1->4: 0.7677   4->1: 0.8097   mean: 0.7887
crop networks                1->4: 0.8774   4->1: 0.9516   mean: 0.9145
frame networks               1->4: 0.9065   4->1: 0.9613   mean: 0.9339
crop + frame networks        1->4: 0.9194   4->1: 0.9742   mean: 0.9468


## Prediction and Export

The crop and frame networks are retrained on all training tracks (sessions 1 and 4), and the same preprocessing (features and scalers fitted on the training set) is applied to the test set. Each test track is classified independently of the others. The predictions are saved in `Es7Predictions.txt`, one digit per line.

In [7]:
proba = (crop_proba(train_m, train_y, test_m, CROPS) + frame_proba(train_m, train_y, test_m)) / 2
predictions = proba.argmax(axis=1)

assert len(predictions) == 620
print('Predicted tracks per digit:', np.bincount(predictions, minlength=10))
save_predictions(predictions, result_path)

Predicted tracks per digit: [60 60 60 59 51 63 65 73 69 60]
Predictions successfully saved to ../Es7Predictions.txt
